In [4]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

pd.set_option('display.width', 140)

TIME_SLOTS = {
    'night_late': {'start': '00:00', 'end': '06:00', 'id': 0},
    'early_morning': {'start': '06:00', 'end': '09:00', 'id': 1},
    'work_morning': {'start': '09:00', 'end': '12:00', 'id': 2},
    'lunch_break': {'start': '12:00', 'end': '15:00', 'id': 3},
    'work_afternoon': {'start': '15:00', 'end': '18:00', 'id': 4},
    'evening_prime': {'start': '18:00', 'end': '21:00', 'id': 5},
    'night_wind': {'start': '21:00', 'end': '23:59', 'id': 6}
}

def assign_time_slot(timestamp):
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)
    ts_time = timestamp.time()
    for slot_name, slot_info in TIME_SLOTS.items():
        start_time = datetime.strptime(slot_info['start'], '%H:%M').time()
        end_time = datetime.strptime(slot_info['end'], '%H:%M').time()
        if slot_info['id'] < 6:
            if start_time <= ts_time < end_time:
                return slot_info['id']
        else:
            if start_time <= ts_time <= end_time:
                return slot_info['id']
    return -1

# ----------------------------------------------------------------
# 1. Charger et préparer
# ----------------------------------------------------------------
df = pd.read_csv("../data/prepared/resultPreparation.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

NUMERIC_FEATURES = ['hour', 'day_of_week', 'month', 'message_length',
                     'has_hashtag', 'has_link', 'has_emoji']
BOOL_FEATURES = ['is_weekend', 'is_business_hours']
CATEGORICAL_FEATURES = ['entreprise', 'content_categorie', 'time_slot_id']
FEATURES = NUMERIC_FEATURES + BOOL_FEATURES + CATEGORICAL_FEATURES
TARGET = 'total_engagement'

model_df = df[FEATURES + [TARGET, 'timestamp']].copy()
model_df['year'] = model_df['timestamp'].dt.year
model_df = model_df.dropna(subset=FEATURES + [TARGET])
for c in BOOL_FEATURES:
    model_df[c] = model_df[c].astype(int)
model_df['content_categorie'] = model_df['content_categorie'].fillna('unknown')

baseline_norm = model_df.groupby(['entreprise', 'year'])[TARGET].transform('median')
baseline_norm = baseline_norm.replace(0, np.nan)
model_df['engagement_relative'] = model_df[TARGET] / baseline_norm
model_df = model_df.dropna(subset=['engagement_relative'])
model_df['target_log'] = np.log1p(model_df['engagement_relative'])
model_df = model_df.reset_index(drop=True)

X = model_df[FEATURES]
y_log = model_df['target_log']
y_raw = model_df['engagement_relative']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
], remainder='passthrough')

# ----------------------------------------------------------------
# 2. Deux configurations à comparer
# ----------------------------------------------------------------
configs = {
    'Manuel (baseline)': dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               random_state=42, n_jobs=-1),
    'Tuné (RandomizedSearchCV)': dict(n_estimators=500, max_depth=8, learning_rate=0.01,
                                       subsample=0.7, colsample_bytree=1.0,
                                       min_child_weight=3, reg_alpha=0, reg_lambda=0.5,
                                       random_state=42, n_jobs=-1),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = {name: {'r2_raw': [], 'rmse_raw': [], 'mae_raw': [],
                   'r2_log': [], 'rmse_log': [], 'mae_log': []} for name in configs}

print("[1] Validation croisée 5-fold sur les 2 configurations...\n")

for fold_i, (train_idx, test_idx) in enumerate(cv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_log, y_test_log = y_log.iloc[train_idx], y_log.iloc[test_idx]
    y_test_raw = y_raw.iloc[test_idx]

    for name, params in configs.items():
        pipe = Pipeline([('prep', preprocessor), ('model', xgb.XGBRegressor(**params))])
        pipe.fit(X_train, y_train_log)
        pred_log = pipe.predict(X_test)
        pred_raw = np.clip(np.expm1(pred_log), 0, None)

        results[name]['r2_raw'].append(r2_score(y_test_raw, pred_raw))
        results[name]['rmse_raw'].append(np.sqrt(mean_squared_error(y_test_raw, pred_raw)))
        results[name]['mae_raw'].append(mean_absolute_error(y_test_raw, pred_raw))
        results[name]['r2_log'].append(r2_score(y_test_log, pred_log))
        results[name]['rmse_log'].append(np.sqrt(mean_squared_error(y_test_log, pred_log)))
        results[name]['mae_log'].append(mean_absolute_error(y_test_log, pred_log))

    print(f"  Fold {fold_i+1}/5 terminé")

# ----------------------------------------------------------------
# 3. Tableau récapitulatif clair
# ----------------------------------------------------------------
print("\n" + "="*90)
print("RÉSULTATS DÉTAILLÉS PAR FOLD (échelle brute, R²)")
print("="*90)
for name in configs:
    vals = results[name]['r2_raw']
    print(f"{name:35s}: {['%.3f'%v for v in vals]}")

print("\n" + "="*90)
print("MOYENNE ± ÉCART-TYPE SUR LES 5 FOLDS")
print("="*90)
summary_rows = []
for name in configs:
    r = results[name]
    summary_rows.append({
        'Configuration': name,
        'R² brut (moy)': np.mean(r['r2_raw']),
        'R² brut (std)': np.std(r['r2_raw']),
        'RMSE brut (moy)': np.mean(r['rmse_raw']),
        'MAE brut (moy)': np.mean(r['mae_raw']),
        'R² log (moy)': np.mean(r['r2_log']),
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Test de significativité simple : différence moyenne vs écart-type inter-fold
diff = np.array(results['Tuné (RandomizedSearchCV)']['r2_raw']) - np.array(results['Manuel (baseline)']['r2_raw'])
print(f"\nDifférence (Tuné - Manuel) par fold: {['%.3f'%d for d in diff]}")
print(f"Différence moyenne: {diff.mean():.4f} (écart-type: {diff.std():.4f})")
print(f"Le tuné gagne sur {sum(diff>0)}/5 folds")

print("\n✓ Comparaison terminée")

[1] Validation croisée 5-fold sur les 2 configurations...

  Fold 1/5 terminé
  Fold 2/5 terminé
  Fold 3/5 terminé
  Fold 4/5 terminé
  Fold 5/5 terminé

RÉSULTATS DÉTAILLÉS PAR FOLD (échelle brute, R²)
Manuel (baseline)                  : ['0.391', '0.139', '0.005', '0.074', '0.301']
Tuné (RandomizedSearchCV)          : ['0.406', '0.142', '0.005', '0.068', '0.447']

MOYENNE ± ÉCART-TYPE SUR LES 5 FOLDS
            Configuration  R² brut (moy)  R² brut (std)  RMSE brut (moy)  MAE brut (moy)  R² log (moy)
        Manuel (baseline)       0.181888       0.143105        43.903283        4.523533      0.165996
Tuné (RandomizedSearchCV)       0.213766       0.179484        43.209617        4.507795      0.171072

Différence (Tuné - Manuel) par fold: ['0.016', '0.003', '0.000', '-0.006', '0.146']
Différence moyenne: 0.0319 (écart-type: 0.0575)
Le tuné gagne sur 4/5 folds

✓ Comparaison terminée
